<img src="images/logo.png" width=180, align="center"/>

Master's degree in Intelligent Systems

Subject: 11754 - Deep Learning

Year: 2025-2026

Professor: Miguel Ángel Calafat Torrens

# LESSON 4A — Diffusion Models: Theory & Architecture

**Note: The code of this example is taken from [this repo](https://github.com/dome272/Diffusion-Models-pytorch) with license Apache 2.0 ([license](https://github.com/dome272/Diffusion-Models-pytorch/blob/main/LICENSE)), so this is also the license of this document.**

**You are strongly encouraged to visit the repo and watch the explanatory videos:**
- [Diffusion Models | Paper Explanation | Math Explained](https://www.youtube.com/watch?v=HoKDTa5jHvg)
- [Diffusion Models | Pytorch Implementation](https://www.youtube.com/watch?v=TBCRlnwJtZU)

**Watch the videos before working through this notebook.** They provide an excellent visual explanation of the concepts we cover here. This notebook complements those videos with formulas, code, and hands-on visualizations.

In [ ]:
# Environment detection: Colab vs local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running on {"Google Colab" if IN_COLAB else "local environment"}')

In [ ]:
# Setup: Drive mount (Colab) or local path
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    %cd '/content/gdrive/MyDrive/LABS2026/LAB04'

import os
import pathlib
import sys

PROJECT_DIR = str(pathlib.Path().resolve())
sys.path.append(PROJECT_DIR)

import helper_L4 as hp

import logging
import torch
from matplotlib import pyplot as plt
import numpy as np

logging.basicConfig(format="%(asctime)s - %(levelname)s: %(message)s",
                    level=logging.INFO, datefmt="%I:%M:%S")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 1. From GANs to Diffusion Models

In LAB03 we explored **Generative Adversarial Networks (GANs)** — two networks in an adversarial game. GANs can produce sharp images quickly (single forward pass), but suffer from training instability and mode collapse.

**Diffusion models** take a fundamentally different approach:

| | GANs | Diffusion Models |
|---|---|---|
| **Core idea** | Adversarial game (generator vs discriminator) | Iterative denoising |
| **Training** | Minimax optimization (unstable) | Simple MSE loss (stable) |
| **Sampling** | Single forward pass (fast) | T iterative steps (slow) |
| **Quality** | Sharp but less diverse | High quality and diverse |
| **Mode coverage** | Risk of mode collapse | Full distribution coverage |

Diffusion models are the backbone of modern systems like Stable Diffusion and DALL-E. Their key insight: instead of generating images in one shot, **gradually denoise random noise into structured output** over many small steps.

The diffusion process has two phases: a **forward process** that progressively adds noise to data, and a **reverse process** that learns to undo the noise.

<img src="images/diffusion_overview.png" width=1100, align="center"/>

Image source: https://cvpr2022-tutorial-diffusion-models.github.io/

# 2. The Forward Diffusion Process

## Adding Noise Step by Step

The forward process gradually adds Gaussian noise to a clean image $x_0$ over $T$ timesteps. At each step:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\left(\sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t\, I\right)$$

where $\beta_t$ is the **noise schedule** — a small value that controls how much noise is added at each step. We define:
- $\beta_t$: noise variance at step $t$ (the schedule)
- $\alpha_t = 1 - \beta_t$: signal retention at step $t$
- $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$: cumulative signal retention up to step $t$

## The Reparameterization Trick (Key Insight)

A crucial property of Gaussian noise: we can skip directly to **any** timestep $t$ without iterating through all intermediate steps:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon \qquad \text{where } \epsilon \sim \mathcal{N}(0, I)$$

This is what makes training efficient — we can sample a random timestep $t$ and compute $x_t$ directly from $x_0$ in a single operation.

## Understanding $\bar{\alpha}_t$

The cumulative product $\bar{\alpha}_t$ controls how much of the original signal remains at timestep $t$:

- At $t = 0$: $\bar{\alpha}_0 \approx 1$ — the image is almost untouched (mostly signal)
- At $t = T$: $\bar{\alpha}_T \approx 0$ — the image is almost pure noise

The formula $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$ is a weighted sum: $\sqrt{\bar{\alpha}_t}$ controls the signal weight and $\sqrt{1 - \bar{\alpha}_t}$ controls the noise weight.

In [ ]:
# Visualize the noise schedule parameters
diffusion = hp.Diffusion(noise_steps=1000, img_size=64, device='cpu')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(diffusion.beta.numpy())
axes[0].set_title(r'$\beta_t$ (noise schedule)')
axes[0].set_xlabel('Timestep t')

axes[1].plot(diffusion.alpha.numpy())
axes[1].set_title(r'$\alpha_t = 1 - \beta_t$')
axes[1].set_xlabel('Timestep t')

axes[2].plot(diffusion.alpha_hat.numpy())
axes[2].set_title(r'$\bar{\alpha}_t$ (cumulative product)')
axes[2].set_xlabel('Timestep t')
axes[2].set_ylabel('Signal remaining')

plt.tight_layout()
plt.show()

print(f"At t=0:   alpha_hat = {diffusion.alpha_hat[0]:.4f} (mostly signal)")
print(f"At t=500: alpha_hat = {diffusion.alpha_hat[500]:.4f}")
print(f"At t={diffusion.noise_steps-1}: alpha_hat = {diffusion.alpha_hat[-1]:.6f} (mostly noise)")

# 3. Visualizing the Forward Process

Let's see the forward diffusion in action on real images. We use a subset of the [Landscape Pictures](https://www.kaggle.com/datasets/arnaud58/landscape-pictures) dataset from Kaggle (the same dataset used in the [reference repo](https://github.com/dome272/Diffusion-Models-pytorch)).

**Dataset setup:** Download the dataset from [Kaggle](https://www.kaggle.com/datasets/arnaud58/landscape-pictures), select 1000 images, compress them into a zip file (e.g., `landscape_pictures_1000.zip`), and place it in the datasets folder. The notebook will handle extraction automatically.

In [ ]:
# Dataset setup: extract from zip if needed
if IN_COLAB:
    dataset_zip = '/content/gdrive/MyDrive/datasets/landscape_pictures_1000.zip'
else:
    dataset_zip = os.path.join(PROJECT_DIR, '..', 'datasets', 'landscape_pictures_1000.zip')

DATASET_PATH = hp.extract_dataset(dataset_zip, remove_zip=IN_COLAB)
dataloader = hp.get_data_flat(DATASET_PATH, image_size=64, batch_size=4)

In [ ]:
diffusion = hp.Diffusion(img_size=64, device=DEVICE)
hp.visualize_forward_diffusion(diffusion, dataloader, DEVICE)

Notice how the image progressively degrades. At $t \approx 444$ the image is barely recognizable. By $t = 999$ it's indistinguishable from pure Gaussian noise. The noise schedule controls this degradation rate — we'll explore different schedules in LESSON 4B.

# 4. The UNet Architecture

## Original UNet

The UNet was originally designed for biomedical image segmentation (Ronneberger et al., 2015). Its U-shaped structure combines an encoder-decoder architecture with **skip connections**.

<img src="images/noise_schedule.png" width=600>

*Source: Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional Networks for Biomedical Image Segmentation.*

The **encoder** (contracting path) captures context through progressive downsampling. Each step doubles the number of feature channels while halving spatial resolution.

The **decoder** (expanding path) reconstructs spatial dimensions, restoring fine details.

**Skip connections** link corresponding encoder and decoder layers, concatenating high-resolution features from the encoder with upsampled features in the decoder. This preserves fine-grained spatial details that would otherwise be lost during downsampling. In diffusion models, these connections are critical for preserving the structure needed for accurate denoising.

## UNet with Self-Attention for Diffusion

For diffusion models, the original UNet is modified with self-attention layers and time conditioning. The architecture used here is described in detail in the [implementation video](https://www.youtube.com/watch?v=TBCRlnwJtZU).

<img src="images/unet_arch.png" width=700, align="center"/>

## Time Embedding

The model needs to know **which timestep** it's denoising at. A noisy image at $t=100$ looks very different from one at $t=900$, and the model must behave differently.

The solution: **sinusoidal positional encoding** (the same idea used in Transformers). The scalar timestep $t$ is projected into a high-dimensional vector using sine and cosine functions at different frequencies, then injected into each Down/Up block via an embedding layer.

## Building Blocks

The UNet in `helper_L4.py` is built from these components:

- **DoubleConv**: Two 3×3 convolutions with GroupNorm and GELU activation. Optional residual connection.
- **Down**: MaxPool → DoubleConv (residual) → DoubleConv + time embedding injection. Halves spatial resolution.
- **Up**: Upsample → concatenate skip connection → DoubleConv (residual) → DoubleConv + time embedding. Doubles spatial resolution.
- **SelfAttention**: Multi-head attention applied at specific resolutions. In the encoder: 32×32, 16×16, 8×8. In the decoder: 16×16, 32×32, 64×64. Captures global dependencies at each level.

See `helper_L4.py` for the full implementation of each block.

In [ ]:
# Instantiate the UNet and inspect its structure
model = hp.UNet(device=DEVICE).to(DEVICE)
print(model)

# 5. The Reverse Diffusion Process

## Learning to Denoise

The goal of the reverse process is to learn how to undo the noise. Starting from pure Gaussian noise $x_T$, the model learns the distribution:

$$p_\theta(x_{t-1} \mid x_t)$$

That is, given a noisy image at step $t$, predict the slightly **less noisy** version at step $t-1$.

## The Sampling Formula

At each reverse step, we compute:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}}\, \epsilon_\theta(x_t, t) \right) + \sqrt{\beta_t}\, z$$

where $z \sim \mathcal{N}(0, I)$ for $t > 1$ and $z = 0$ for $t = 1$.

Breaking this down:
- $\frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta \right)$: **signal recovery** — removes the predicted noise contribution
- $\sqrt{\beta_t}\, z$: **stochastic variation** — adds small random noise (except at the final step)
- $\epsilon_\theta(x_t, t)$: the UNet's **noise prediction** at timestep $t$

## Why Predict Noise?

The model predicts $\epsilon$ (the noise that was added) rather than $x_0$ (the clean image) directly. Why?

**Intuition:** The noise always has the same distribution $\mathcal{N}(0, I)$ regardless of the timestep. This makes it a more consistent and stable target for the network to learn. The training loss is simply:

$$L = \| \epsilon - \epsilon_\theta(x_t, t) \|^2$$

A straightforward MSE between the actual noise and the predicted noise. No adversarial games, no balancing act — just simple regression.

In [ ]:
# The sample() method implements the reverse process formula.
# Let's look at its core loop annotated with the formula:

# Starting from pure noise: x ~ N(0, I)
# For each timestep from T-1 down to 1:
#   predicted_noise = model(x, t)          # epsilon_theta(x_t, t)
#   alpha     = alpha[t]
#   alpha_hat = alpha_hat[t]               # cumulative product
#   beta      = beta[t]
#
#   x = (1/sqrt(alpha)) * (x - (1-alpha)/sqrt(1-alpha_hat) * predicted_noise)
#       + sqrt(beta) * z
#
# where z ~ N(0,I) if t > 1, else z = 0

# See hp.Diffusion.sample() for the full implementation
import inspect
print(inspect.getsource(hp.Diffusion.sample))

# 6. Summary

Key takeaways from this notebook:

1. **Forward process**: Gradually adds Gaussian noise over $T$ steps. Thanks to the reparameterization trick, we can jump to any timestep $t$ directly: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon$

2. **Reverse process**: A UNet learns to predict the noise $\epsilon_\theta(x_t, t)$ at each step, enabling iterative denoising from pure noise to a generated image.

3. **UNet architecture**: Encoder-decoder with skip connections, self-attention, and time conditioning. Predicts noise at each timestep.

4. **Training objective**: Simple MSE loss $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$ — much more stable than the adversarial GAN objective.

In **LESSON 4B** we'll cover: training, noise schedules (linear, cosine, sigmoid), learning rate schedulers, and inference with pretrained models.